In [1]:
import pandas as pd, json, numpy as np
from collections import Counter
from collections import defaultdict


gdf = pd.read_parquet("../data/interim/pois.parquet")
SKIP = {"id","tags","visible","version","timestamp","changeset","geometry","osm_type","lon","lat"}

def parse(t):
    if isinstance(t, dict): return t
    if not isinstance(t, str) or not t: return {}
    try: return json.loads(t)
    except Exception: return {}

def attrs(row):
    d = parse(row.get("tags"))
    for c, v in row.items():
        if c in SKIP or v is None or (isinstance(v, float) and np.isnan(v)) or v == "": continue
        d[c] = v
    return d

gdf["attrs"] = gdf.apply(attrs, axis=1)
N = len(gdf)
cov = Counter(k for d in gdf["attrs"] for k in d)
uniq = defaultdict(set)
for d in gdf["attrs"]:
    for k, v in d.items():
        uniq[k].add(str(v))
ktab = pd.DataFrame({"n": cov}).assign(pct=lambda x: 100*x.n/N).sort_values("n", ascending=False)
ktab["n_unique"] = [len(uniq[k]) for k in ktab.index]
print(len(ktab), "ključeva"); print(ktab.head(40))

988 ključeva
                         n        pct  n_unique
amenity              30226  68.906873       138
name                 23740  54.120597     17615
shop                  9580  21.839736       165
building              7908  18.028041        67
addr:street           7468  17.024963      2102
addr:postcode         6167  14.059045       280
addr:city             6142  14.002052       369
addr:housenumber      5594  12.752764       514
religion              4184   9.538356        12
tourism               4177   9.522398        24
opening_hours         4058   9.251111      1302
phone                 3251   7.411376      3023
website               2903   6.618033      2071
denomination          2799   6.380942        28
operator              2774   6.323948      1135
parking               2426   5.530605        14
access                2140   4.878605        10
name:sr               1874   4.272199      1689
brand                 1799   4.101220       338
fee                   1660 

Ćelija iznad je za svaki red pravila attr kolonu sa svim atributima, koja uključuje merganu tags kolonu sa posebnim tagovima i sve neprazne kolone koje su osm tagovi. Isključene su metapodaci koji su navedeni u listi. Izračunato je da ima čak 988 jedinstvenih ključeva, od kojih amenity pokriva 68 posto svih redova, međutim, samo prvih 8 pokriva preko 10 posto redova tabele.

In [3]:
order = ktab.index.tolist()
total_pairs = ktab.n.sum()
rows = []
for k in [5,10,15,20,30,50,100,len(order)]:
    keep = set(order[:k]) | {"name"}
    kept = ktab.loc[list(keep), "n"].sum()
    empty = (gdf["attrs"].map(lambda d: len(keep & set(d)) == 0)).mean()
    rows.append((k, kept/total_pairs, empty))
print(pd.DataFrame(rows, columns=["k","pairs_retained","rows_with_no_attr"]))

     k  pairs_retained  rows_with_no_attr
0    5        0.458316           0.026673
1   10        0.610836           0.000000
2   15        0.702503           0.000000
3   20        0.759988           0.000000
4   30        0.826225           0.000000
5   50        0.888368           0.000000
6  100        0.943897           0.000000
7  988        1.000000           0.000000

Ovaj cell uzima sve parove, i broji koliko parova od ukupno key-values parova ostaje u attr ako se uzme samo k najčešćih atributa, te paralelno broji koliko redova bi ostalo bez ikakvog atributa. Na 20 najčešćih atributa, a što je već mnogo kolona, vidimo da bi ostali na 25 posto izgubljenih informacija.

In [4]:
descr = [k for k in order if k not in {"ref","url","phone","email","website","source","wikipedia"}
         and ktab.loc[k,"n_unique"] > 1]
covered = np.zeros(N, bool); picked = []
masks = {k: gdf["attrs"].map(lambda d, k=k: k in d).to_numpy() for k in descr[:80]}
for _ in range(15):
    gain = {k: (m & ~covered).sum() for k, m in masks.items() if k not in picked}
    if not gain or max(gain.values()) == 0: break
    k = max(gain, key=gain.get); picked.append(k); covered |= masks[k]
    print(k, gain[k], f"{covered.mean():.1%}")

amenity 30226 68.9%
name 10446 92.7%
shop 1842 96.9%
tourism 1351 100.0%


Pohlepni algoritam računa koliko redova se pokriva minimalnim brojem kolona i koje su kolone. To su ove gore navedene kolone

In [12]:
def txt(d): return " ".join(f"{k} {v}" for k,v in d.items())
lens = gdf["attrs"].map(lambda d: len(txt(d).split()))
print(lens.describe(percentiles=[.5,.9,.99]))

count    43865.000000
mean         9.738448
std         10.518400
min          2.000000
50%          6.000000
90%         22.000000
99%         47.000000
max        423.000000
Name: attrs, dtype: float64


Statistika attrs u smislu ukupnog broja riječi-i key i value su zasebna riječ. 99 posto POI-a ima manje ili jednako 47 riječi. Najduža zabilježena riječ je 423 riječi. Ako value ima više riječi onda i svaka ta pojedinačna riječ ulazi u broj

In [13]:
gdf.shape

(43865, 88)

In [14]:
gdf.columns.tolist()    # imena svih kolona
gdf.info()              # kolone, tipovi i broj nepraznih vrijednosti po koloni
gdf.notna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 43865 entries, 0 to 43864
Data columns (total 88 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                43865 non-null  int64  
 1   lon               27017 non-null  float64
 2   lat               27017 non-null  float64
 3   tags              19247 non-null  str    
 4   visible           43865 non-null  bool   
 5   version           43865 non-null  int32  
 6   timestamp         43865 non-null  uint32 
 7   changeset         27139 non-null  float64
 8   addr:city         6142 non-null   str    
 9   addr:country      359 non-null    str    
 10  addr:housenumber  5594 non-null   str    
 11  addr:postcode     6167 non-null   str    
 12  addr:place        292 non-null    str    
 13  addr:street       7468 non-null   str    
 14  email             1161 non-null   str    
 15  name              23740 non-null  str    
 16  opening_hours     4058 non-null   str    
 17  oper

id               43865
lon              27017
lat              27017
tags             19247
visible          43865
                 ...  
coffee               2
security             1
swimming_pool        2
library              1
attrs            43865
Length: 88, dtype: int64